In [1]:
import pandas as pd
import os

In [2]:
inDIR = '../data/training/'

In [40]:
# which data to use
#prefix = 'plot'
prefix = 'transect'

inFILE = 'vor_2014_2023_cln_2024_04_04_' + prefix + '_hls_idxs.csv'
nickname = 'cper_bm_' + prefix

inPATH = os.path.join(inDIR, inFILE)

# unique ID column name
id_col = 'Id'
# date column name
date_col = 'Date_mean'
# dependent variable column
y_col = 'Biomass_kg_ha'

var_names = [
    'NDVI', 'DFI', 'NDTI', 'SATVI', 'NDII7', 'SAVI',
    'RDVI', 'MTVI1', 'NCI', 'NDCI', 'PSRI', 'NDWI', 'EVI', 'TCBI', 'TCGI', 'TCWI',
    'BAI_126', 'BAI_136', 'BAI_146', 'BAI_236', 'BAI_246', 'BAI_346',
    'BLUE', 'GREEN', 'RED', 'NIR1', 'SWIR1', 'SWIR2'
]


In [41]:
past_attributes = pd.read_csv('../data/ground/boundaries/CARM_PlotAttributes_spk.csv')
past_attributes['Id'] = past_attributes.apply(lambda x: '_'.join([x['PastureCode'], 
                                                                  'P'+str(x['Plot'])]),
                                              axis=1)

df = pd.read_csv(inPATH, parse_dates=[date_col])

df = df[df['Season'].isin(['June', 'October'])].copy()

df = df[~df[var_names + [y_col]].isnull().any(axis=1)].copy()

df['Plot'] = df[id_col].apply(lambda x: x.split('_')[1])
if prefix == 'transect':
    df['Transect'] = df[id_col].apply(lambda x: x.split('_')[2])
    df['Id_tmp'] = df[id_col].apply(lambda x: '_'.join(x.split('_')[:-1]))
    df = pd.merge(df, past_attributes[['Id', 'Ecosite', 'Spatial']], left_on='Id_tmp', right_on='Id', how='left')
    df = df.rename(columns={'Id_x': 'Id'})
    df = df.drop(columns=['Id_y', 'Id_tmp'])
else:
    df = pd.merge(df, past_attributes[['Id', 'Ecosite', 'Spatial']], on='Id', how='left')
df = df.rename(columns={'Ecosite': 'ecosite',
                        'Spatial': 'spatial'})
df = df[df['ecosite'] != 'Overflow']

In [43]:
if prefix == 'transect':
    df = df[['Id', 'Pasture', 'Plot', 'Transect', 'ecosite', 'spatial',
         'Date', 'Date_mean', 'Year', 'Season', 'Low', 'High',
         'Biomass_kg_ha', 'geometry'] + var_names]
else:
    df = df[['Id', 'Pasture', 'Plot', 'ecosite', 'spatial',
             'Date', 'Date_mean', 'Year', 'Season', 'Low', 'High',
             'Biomass_kg_ha', 'geometry'] + var_names]

In [44]:
df.to_csv('../data/final/kearney_scirep_' + prefix + '_fnl.csv', index=False)